# KNN (K-Nearest-Neighbors)

In [1]:
import numpy as np
import pandas as pd

# aqui eu carrego as notas que o pessoal deu pros filmes
r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('Aquivos_de_Codigo_LAMIA/u.data',
                      sep='\t', names=r_cols, usecols=range(3))
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [2]:
# agrupo por filme e pego quantas notas ele tem e qual a media
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})
movieProperties.head()

rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [3]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
# deixo a quantidade de notas entre 0 e 1 pra virar uma nota de popularidade
movieNormalizeNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x))/(np.max(x) - np.min(x)))
movieNormalizeNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


In [4]:
movieDict = { }
# aqui eu monto um catalogo com nome genero popularidade e nota de cada filme
# a base e de 1998 e vem em latin-1 entao preciso dizer isso aqui
# sem o encoding ela ate abre no Windows mas quebra no Linux
with open (r'Aquivos_de_Codigo_LAMIA/u.item', encoding='latin-1') as f:
    temp = ''
    for line in f:
        fields = line.rstrip('\n').split('|')
        movieID = int(fields[0])
        name = fields[1]
        genres = fields[5:25]
        # viro genero em numero senao o ComputeDistance quebra
        genres = list(map(int, genres))
        movieDict[movieID] = (name, genres, movieNormalizeNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating)

In [5]:
movieDict[1] # dou uma olhada no filme 1 que e o Toy Story

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)

In [6]:
from scipy import spatial

# minha distancia e a diferenca de genero somada com a diferenca de popularidade
def ComputeDistance(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB)
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)
    return genreDistance + popularityDistance

# quanto menor o numero mais parecidos os dois filmes sao
ComputeDistance(movieDict[2], movieDict[4])

np.float64(0.8004574042309892)

In [7]:
# imprimo os dois pra ver se aquela distancia faz sentido
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), size    131.000000
mean      3.206107
Name: 2, dtype: float64)
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), size    209.000000
mean      3.550239
Name: 4, dtype: float64)


In [8]:
import operator

# essa funcao mede a distancia pra todos os filmes e devolve os K mais perto
def getNeighbors(movieID, K):
    distances = []

    for movie in movieDict:
        if (movie != movieID):
            dist = ComputeDistance(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))

    distances.sort(key=operator.itemgetter(1))

    neighbors = []

    for x in range(K):
        neighbors.append(distances[x][0])

    return neighbors

K = 10
avgRating = 0
# pego os 10 vizinhos do Toy Story
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3]
    print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))

# tiro a media das notas desses 10
avgRating /= float(K)

Liar Liar (1997) size    485.000000
mean      3.156701
Name: 294, dtype: float64
Aladdin (1992) size    219.000000
mean      3.812785
Name: 95, dtype: float64
Willy Wonka and the Chocolate Factory (1971) size    326.000000
mean      3.631902
Name: 151, dtype: float64
Monty Python and the Holy Grail (1974) size    316.000000
mean      4.066456
Name: 168, dtype: float64
Full Monty, The (1997) size    315.000000
mean      3.926984
Name: 269, dtype: float64
George of the Jungle (1997) size    162.000000
mean      2.685185
Name: 259, dtype: float64
Beavis and Butt-head Do America (1996) size    156.000000
mean      2.788462
Name: 240, dtype: float64
Birdcage, The (1996) size    293.000000
mean      3.443686
Name: 25, dtype: float64
Home Alone (1990) size    137.000000
mean      3.087591
Name: 94, dtype: float64
Aladdin and the King of Thieves (1996) size    26.000000
mean     2.846154
Name: 422, dtype: float64


In [9]:
avgRating # esse e o palpite do KNN pra nota do Toy Story

size    243.500000
mean      3.344591
Name: 294, dtype: float64

In [10]:
movieDict[1] # aqui a nota real pra eu comparar com o palpite

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)